# @toolを使わずにツールを定義する

## 1、例

In [1]:
# 1、モデルの初期化
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print

# .envファイルから環境変数を読み込む
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
DEEPSEEK_MODEL = os.getenv("DEEPSEEK_MODEL")

model = init_chat_model(
    model=DEEPSEEK_MODEL,
    api_key=DEEPSEEK_API_KEY,
)

# 2、関数（ツール）を宣言
def get_weather(city : str):
    return f"{city}は晴れです~~"

# 3、関数をモデルにバインド
model_with_tools = model.bind_tools([get_weather])

# 4、モデルを呼び出し
response = model_with_tools.invoke("北京の天気はどうですか")
print(response)

AIMessage(
    content='',
    additional_kwargs={
        'refusal': None,
        'reasoning_content': '用户想知道北京的天气。我需要使用get_weather工具，参数是city为"北京"。'
    },
    response_metadata={
        'token_usage': {
            'completion_tokens': 64,
            'prompt_tokens': 273,
            'total_tokens': 337,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 19,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 256},
            'prompt_cache_hit_tokens': 256,
            'prompt_cache_miss_tokens': 17
        },
        'model_provider': 'deepseek',
        'model_name': 'deepseek-v4-pro',
        'system_fingerprint': 'fp_9954b31ca7_prod0820_fp8_kvcache_20260402',
        'id': '5b5634f1-ff4e-44cd-b2ac-83e5f31ca9f0',
        'finish_reason': 'tool_calls',
        'logprobs': None
    },
    id='lc_run--019fc16f-027a-7c43-8174-7de1b52aefa6-0',
    tool_calls=[
        {
            'name': 'get_weather',
            'args': {'city': '北京'},
            'id': 'call_00_S5fqon1Bt9O08IEG3Gkh2422',
            'type': 'tool_call'
        }
    ],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 273,
        'output_tokens': 64,
        'total_tokens': 337,
        'input_token_details': {'cache_read': 256},
        'output_token_details': {'reasoning': 19}
    }
)

## 2、ツール説明の各部分の詳細

## 2.1 convert_to_openai_toolを理解する

`model.bind_tools([get_weather])`を実行すると、内部的には最終的に`convert_to_openai_tool`を呼び出してツールの説明を生成します。そのため、後者を直接呼び出してパース後のツール説明を確認することができます。

In [2]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    return f"{city}は晴れです~~"


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 2.2 description の説明

In [3]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    都市の天気を検索する
    
    Args:
        city:具体的な都市
    """
    return f"{city}は晴れです~~"


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を検索する',
        'parameters': {
            'properties': {'city': {'description': '具体的な都市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.3 パラメータの説明

In [4]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str):
    """
    都市の天気を検索する

    Args:
        city : 具体的な都市

    Returns:
        都市の天気を返す
    """
    return f"{city}は晴れです~~"


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を検索する',
        'parameters': {
            'properties': {'city': {'description': '具体的な都市', 'type': 'string'}},
            'required': ['city'],
            'type': 'object'
        }
    }
}

## 2.4 パラメータの型の説明

例1：正しい例

In [5]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    都市の天気を検索する
    """
    return f"{city}は晴れです~~"


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を検索する',
        'parameters': {'properties': {'city': {}}, 'required': ['city'], 'type': 'object'}
    }
}

例2：以下のコードは実行するとエラーになります

要件：docstring内でパラメータの説明を宣言する場合、関数宣言側でパラメータの型を明示する必要があります

In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city):
    """
    都市の天気を検索する

    Args:
        city : 具体的な都市
    """
    return f"{city}は晴れです~~"


print(convert_to_openai_tool(get_weather))

ValueError: Arg city in docstring not found in function signature.

## 2.5 パラメータのデフォルト値の説明

パラメータにデフォルト値を設定すると、出力結果の required フィールドにこのパラメータは含まれなくなります。

例1：

In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(city : str = "beijing"):
    """
    都市の天気を検索する

    Args:
        city : 具体的な都市
    """
    return f"{city}は晴れです~~"


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を検索する',
        'parameters': {
            'properties': {'city': {'default': 'beijing', 'description': '具体的な都市', 'type': 'string'}},
            'type': 'object'
        }
    }
}

例2：

In [8]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def get_weather(dt:str ,city : str = "beijing"):
    """
    都市の天気を検索する

    Args:
        city : 具体的な都市
        dt : 日時
    """
    return f"{city}は晴れです~~"


print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '都市の天気を検索する',
        'parameters': {
            'properties': {
                'dt': {'description': '日時', 'type': 'string'},
                'city': {'default': 'beijing', 'description': '具体的な都市', 'type': 'string'}
            },
            'required': ['dt'],
            'type': 'object'
        }
    }
}